Creating Dimension Tables

In [0]:
# Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS final_project.gold")

Product Dimension

In [0]:
# Dim_Product
df_silver_product = spark.read.table("final_project.silver.product")

# Select only the specific columns
df_silver_dim_product = df_silver_product.select(
    "product_key",
    "product",
    "standard_cost",
    "color",
    "subcategory",
    "category"
)

# Write to Gold
df_silver_dim_product.write.mode("overwrite").saveAsTable("final_project.gold.dim_product")

# Display the Gold table
display(spark.read.table("final_project.gold.dim_product"))

# Change the actual column metadata to NOT NULL (This fixes your error)
spark.sql("ALTER TABLE final_project.gold.dim_product ALTER COLUMN product_key SET NOT NULL")

# Drop with CASCADE to also remove child foreign key constraints
spark.sql("ALTER TABLE final_project.gold.dim_product DROP CONSTRAINT IF EXISTS product_key_pk CASCADE")

# Add the Primary Key
spark.sql("ALTER TABLE final_project.gold.dim_product DROP CONSTRAINT IF EXISTS product_key_pk")
spark.sql("ALTER TABLE final_project.gold.dim_product ADD CONSTRAINT product_key_pk PRIMARY KEY (product_key) RELY")

Region Dimension

In [0]:
# Dim_Region
df_silver_region = spark.read.table("final_project.silver.region")
df_silver_region.write.mode("overwrite").saveAsTable("final_project.gold.dim_region")

# Change the actual column metadata to NOT NULL
spark.sql("ALTER TABLE final_project.gold.dim_region ALTER COLUMN sales_territory_key SET NOT NULL")

# Drop with CASCADE to also remove child foreign key constraints
spark.sql("ALTER TABLE final_project.gold.dim_region DROP CONSTRAINT IF EXISTS sales_territory_key_pk CASCADE")

# Drop all constraints first to avoid "Already Exists" errors
spark.sql("ALTER TABLE final_project.gold.dim_region DROP CONSTRAINT IF EXISTS sales_territory_key_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_region DROP CONSTRAINT IF EXISTS sales_territory_key_pk")

# Add constraints fresh
spark.sql("ALTER TABLE final_project.gold.dim_region ADD CONSTRAINT sales_territory_key_not_null CHECK (sales_territory_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.dim_region ADD CONSTRAINT sales_territory_key_pk PRIMARY KEY (sales_territory_key) RELY")

Reseller Dimension

In [0]:
# Dim_Reseller
df_silver_reseller = spark.read.table("final_project.silver.reseller")
df_silver_reseller.write.mode("overwrite").saveAsTable("final_project.gold.dim_reseller")

# Change the actual column metadata to NOT NULL
spark.sql("ALTER TABLE final_project.gold.dim_reseller ALTER COLUMN reseller_key SET NOT NULL")

# Drop with CASCADE to also remove child foreign key constraints
spark.sql("ALTER TABLE final_project.gold.dim_reseller DROP CONSTRAINT IF EXISTS reseller_key_pk CASCADE")

# Drop all constraints first to avoid "Already Exists" errors
spark.sql("ALTER TABLE final_project.gold.dim_reseller DROP CONSTRAINT IF EXISTS reseller_key_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_reseller DROP CONSTRAINT IF EXISTS reseller_key_pk")

# Add constraints fresh
spark.sql("ALTER TABLE final_project.gold.dim_reseller ADD CONSTRAINT reseller_key_not_null CHECK (reseller_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.dim_reseller ADD CONSTRAINT reseller_key_pk PRIMARY KEY (reseller_key) RELY")

SalesPerson Dimension

In [0]:
# Dim_Salesperson
df_silver_salesperson = spark.read.table("final_project.silver.salesperson")
df_silver_salesperson.write.mode("overwrite").saveAsTable("final_project.gold.dim_salesperson")

# Check the actual column metadata to NOT NULL
spark.sql("ALTER TABLE final_project.gold.dim_salesperson ALTER COLUMN employee_key SET NOT NULL")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson ALTER COLUMN employee_id SET NOT NULL")

# Drop with CASCADE to also remove child foreign key constraints
spark.sql("ALTER TABLE final_project.gold.dim_salesperson DROP CONSTRAINT IF EXISTS employee_key_pk CASCADE")

# Drop all constraints first to avoid "Already Exists" errors
spark.sql("ALTER TABLE final_project.gold.dim_salesperson DROP CONSTRAINT IF EXISTS employee_key_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson DROP CONSTRAINT IF EXISTS employee_id_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson DROP CONSTRAINT IF EXISTS employee_key_pk")

# Add constraints fresh
spark.sql("ALTER TABLE final_project.gold.dim_salesperson ADD CONSTRAINT employee_key_not_null CHECK (employee_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson ADD CONSTRAINT employee_id_not_null CHECK (employee_id IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson ADD CONSTRAINT employee_key_pk PRIMARY KEY (employee_key) RELY")

Creating Date Dimension

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, date_format, quarter, dayofweek

In [0]:
# Generate a continuous sequence of dates
# Adjust the start and end dates to match the overall range of the sales data
df_date_spine = spark.sql("""
    SELECT explode(sequence(
        to_date('2017-01-01'), 
        to_date('2020-12-31'), 
        interval 1 day
    )) as order_date
""")

In [0]:
# Extract all the useful analytical attributes
df_dim_date = df_date_spine.select(
    col("order_date"),                                       # e.g., 2019-05-25
    year("order_date").alias("year"),                        # e.g., 2019
    quarter("order_date").alias("quarter"),                  # e.g., 2
    month("order_date").alias("month_number"),               # e.g., 5
    date_format("order_date", "MMMM").alias("month_name"),   # e.g., May
    dayofmonth("order_date").alias("day_of_month"),          # e.g., 25
    date_format("order_date", "EEEE").alias("day_of_week")   # e.g., Saturday
)

In [0]:
# View the generated calendar
display(df_dim_date)

In [0]:
# Print the schema
df_dim_date.printSchema()

In [0]:
# Save to the Gold layer
df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("final_project.gold.dim_date")

In [0]:
# enforce not null metadata on the key column
spark.sql("ALTER TABLE final_project.gold.dim_date ALTER COLUMN order_date SET NOT NULL")

# Drop with CASCADE to also remove child foreign key constraints
spark.sql("ALTER TABLE final_project.gold.dim_date DROP CONSTRAINT IF EXISTS date_pk CASCADE")

# Drop the constraints first to avoid "Already Exists" errors
spark.sql("ALTER TABLE final_project.gold.dim_date DROP CONSTRAINT IF EXISTS date_pk")

# define the primary key
spark.sql("ALTER TABLE final_project.gold.dim_date ADD CONSTRAINT date_pk PRIMARY KEY(order_date) NOT ENFORCED")

SalesPerson Region Dimension

In [0]:
# Dim_Salesperson_Region (Bridge)
df_silver_salesperson_region = spark.read.table("final_project.silver.salesperson_region")
df_silver_salesperson_region.write.mode("overwrite").saveAsTable("final_project.gold.dim_salesperson_region")

# Drop constraints first to avoid "Already Exists" errors
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region DROP CONSTRAINT IF EXISTS employee_key_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region DROP CONSTRAINT IF EXISTS sales_territory_key_not_null")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region DROP CONSTRAINT IF EXISTS fk_bridge_emp")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region DROP CONSTRAINT IF EXISTS fk_bridge_reg")

# Constraints
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region ADD CONSTRAINT employee_key_not_null CHECK (employee_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region ADD CONSTRAINT sales_territory_key_not_null CHECK (sales_territory_key IS NOT NULL)")

# Adding the Foreign Keys for the Bridge
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region ADD CONSTRAINT fk_bridge_emp FOREIGN KEY(employee_key) REFERENCES final_project.gold.dim_salesperson(employee_key) NOT ENFORCED")
spark.sql("ALTER TABLE final_project.gold.dim_salesperson_region ADD CONSTRAINT fk_bridge_reg FOREIGN KEY(sales_territory_key) REFERENCES final_project.gold.dim_region(sales_territory_key) NOT ENFORCED")

**Fact Tables**

Fact Sales

In [0]:
# Fact_Sales
df_sales_silver = spark.read.table("final_project.silver.sales")
df_sales_silver.write.format("delta").mode("overwrite").saveAsTable("final_project.gold.fact_sales")

# Update Column Metadata (Sets the required NOT NULL status for keys)
spark.sql("ALTER TABLE final_project.gold.fact_sales ALTER COLUMN order_date SET NOT NULL")
spark.sql("ALTER TABLE final_project.gold.fact_sales ALTER COLUMN product_key SET NOT NULL")
spark.sql("ALTER TABLE final_project.gold.fact_sales ALTER COLUMN employee_key SET NOT NULL")
spark.sql("ALTER TABLE final_project.gold.fact_sales ALTER COLUMN sales_territory_key SET NOT NULL")

# DROP all constraints first to avoid "Already Exists" errors
# Drop Check Constraints
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS sales_date_not_null")
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS sales_prod_not_null")
# Corrected your notebook: product_key instead of reseller_key for this specific check
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS sales_emp_not_null")
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS sales_reg_not_null")

# Drop Foreign Key Constraints
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_date")
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_product")
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_employee")
spark.sql("ALTER TABLE final_project.gold.fact_sales DROP CONSTRAINT IF EXISTS fk_sales_territory")

# Add Check Constraints
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT sales_date_not_null CHECK (order_date IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT sales_prod_not_null CHECK (product_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT sales_emp_not_null CHECK (employee_key IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT sales_reg_not_null CHECK (sales_territory_key IS NOT NULL)")

# Add Foreign Keys 
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT fk_sales_date FOREIGN KEY(order_date) REFERENCES final_project.gold.dim_date(order_date) NOT ENFORCED")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT fk_sales_product FOREIGN KEY(product_key) REFERENCES final_project.gold.dim_product(product_key) NOT ENFORCED")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT fk_sales_employee FOREIGN KEY(employee_key) REFERENCES final_project.gold.dim_salesperson(employee_key) NOT ENFORCED")
spark.sql("ALTER TABLE final_project.gold.fact_sales ADD CONSTRAINT fk_sales_territory FOREIGN KEY(sales_territory_key) REFERENCES final_project.gold.dim_region(sales_territory_key) NOT ENFORCED")

Fact Targets

In [0]:
# Fact_Targets
df_targets_silver = spark.read.table("final_project.silver.targets")
df_targets_silver.write.mode("overwrite").saveAsTable("final_project.gold.fact_targets")

# Update Column Metadata
# This changes the schema to NOT NULL so it can support foreign key relationships
spark.sql("ALTER TABLE final_project.gold.fact_targets ALTER COLUMN employee_id SET NOT NULL")
spark.sql("ALTER TABLE final_project.gold.fact_targets ALTER COLUMN target_date SET NOT NULL")

# DROP all existing constraints first to avoid "Already Exists" errors
# Drop Check Constraints
spark.sql("ALTER TABLE final_project.gold.fact_targets DROP CONSTRAINT IF EXISTS target_emp_not_null")
spark.sql("ALTER TABLE final_project.gold.fact_targets DROP CONSTRAINT IF EXISTS target_date_not_null")

# Drop Foreign Key Constraints
spark.sql("ALTER TABLE final_project.gold.fact_targets DROP CONSTRAINT IF EXISTS fk_target_date")
spark.sql("ALTER TABLE final_project.gold.fact_targets DROP CONSTRAINT IF EXISTS fk_target_employee")

# Add Check Constraints
spark.sql("ALTER TABLE final_project.gold.fact_targets ADD CONSTRAINT target_emp_not_null CHECK (employee_id IS NOT NULL)")
spark.sql("ALTER TABLE final_project.gold.fact_targets ADD CONSTRAINT target_date_not_null CHECK (target_date IS NOT NULL)")

# Add Foreign Keys
# This links the Target fact table to your shared Dimensions (Date and Salesperson)
# link to the date dimension
spark.sql("ALTER TABLE final_project.gold.fact_targets ADD CONSTRAINT fk_target_date FOREIGN KEY(target_date) REFERENCES final_project.gold.dim_date(order_date) NOT ENFORCED")

# link to the salesperson/employee dimension
spark.sql("ALTER TABLE final_project.gold.fact_targets ADD CONSTRAINT fk_target_employee FOREIGN KEY(employee_id) REFERENCES final_project.gold.dim_salesperson(employee_key) NOT ENFORCED")

Export Delta Tables into Parquet Files

In [0]:
# Path to your Volume
volume_path = "/Volumes/final_project/gold/export_volume/"

# Automatically get the list of all tables in your Gold schema
gold_tables = spark.catalog.listTables("final_project.gold")

# Loop through every table and save it
for table in gold_tables:
    table_name = table.name
    print(f"Exporting {table_name}...")
    
    df = spark.table(f"final_project.gold.{table_name}")
    
    # Save as a single Parquet file inside a folder named after the table
    df.coalesce(1).write.mode("overwrite").parquet(f"{volume_path}{table_name}")

print("All Gold tables have been exported to the Volume")